In [1]:
# Install Unsloth (fastest for QLoRA) and other required libraries
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qqq --no-deps xformers==0.0.28.post2
!pip install -qqq trl peft accelerate bitsandbytes datasets huggingface_hub
!pip install -qqq openai   # for Qwen/Alibaba Cloud API (OpenAI-compatible)

import torch
from unsloth import FastLanguageModel
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import pandas as pd
from google.colab import drive

# Mount Google Drive so we can save checkpoints safely
drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/CSE4078_Project", exist_ok=True)

print("✅ Setup done!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 160.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 163.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 159.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 121.4 MB/

In [2]:
import os
from datasets import load_from_disk # Changed from load_dataset to load_from_disk

# Define the path where datasets are saved in Google Drive
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"

# Load the pre-processed datasets directly from Drive
train_dataset = load_from_disk(f"{data_save_path}/train_dataset")
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
print("✅ Datasets loaded from disk!")

# Print sizes to confirm
print(f"✅ Final splits \u2192 Train: {len(train_dataset)} | Test (unseen): {len(test_dataset)}")

✅ Datasets loaded from disk!
✅ Final splits → Train: 20000 | Test (unseen): 1999


In [ ]:
# ==================== 3. MODEL + FINE-TUNING ====================
model_name = "Qwen/Qwen3.5-2B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,                    # Auto
    load_in_4bit=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,                          # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Format function (Qwen chat style)
def format_example(example):
    return {
        "text": tokenizer.apply_chat_template(
            [
                {"role": "user", "content": example["question_content"]},
                {"role": "assistant", "content": example["question_answer"]}
            ],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_example, num_proc=4)

# ==================== 4. TRAINER (with checkpointing) ====================
from trl import SFTTrainer
from transformers import TrainingArguments

# ==================== HARDWARE & PRECISION DIAGNOSTICS ====================
print("-" * 50)
print("TRAINING DIAGNOSTICS:")
has_gpu = torch.cuda.is_available()
print(f"GPU Available: {has_gpu}")
if has_gpu:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

is_bf16 = torch.cuda.is_bf16_supported() if has_gpu else False
print(f"Bfloat16 (BF16) Supported: {is_bf16}")
print(f"Using FP16 Fallback: {not is_bf16}")

# Check the data type of the model's parameters
model_dtype = next(model.parameters()).dtype
print(f"Model Base Data Type: {model_dtype}")

# Print LoRA trainable parameters
trainable_params = 0
all_param = 0
for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
print(f"Total Parameters: {all_param:,}")
print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / all_param:.4f}%)")
print("-" * 50)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=4,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,      # adjust if OOM
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=50,                     # ← checkpoint every X steps
        save_total_limit=5,
        output_dir="/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v2",
        push_to_hub=False,
        report_to="none",
        save_strategy="steps",
    ),
)

print("🚀 Starting fine-tuning now...")
trainer.train(resume_from_checkpoint=False)   # change to True if resuming

==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


🚀 Starting fine-tuning now...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,000 | Num Epochs = 5 | Total steps = 1,565
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 21,823,488 of 2,235,065,152 (0.98% trained)


Step,Training Loss
10,3.911833
20,3.506596
30,3.217801
40,3.131289
50,3.011988
60,2.926572
70,2.820030
80,2.811852
90,2.699644
100,2.713653


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v2/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v2/checkpoint-100/tokenizer_config.json.


### Key Parameters for Better Fine-tuning

There are several important parameters you can adjust in the `FastLanguageModel.get_peft_model` function and `TrainingArguments` to optimize your fine-tuning results:

#### 1. LoRA Parameters (`FastLanguageModel.get_peft_model`)

*   **`r` (LoRA rank)**: This determines the dimensionality of the low-rank matrices used in LoRA. A higher `r` means more trainable parameters, which can lead to better performance if your model has enough capacity and data, but also increases memory usage and training time. Common values range from 8 to 64.
    *   **Recommendation**: Start with `32` or `64` and adjust based on performance and available memory.

*   **`lora_alpha`**: This is a scaling factor for the LoRA updates. It controls the magnitude of the updates to the original model weights. A higher `lora_alpha` gives more weight to the LoRA adaptations.
    *   **Recommendation**: Often set equal to `r` or `2 * r`.

*   **`target_modules`**: This specifies which layers of the base model will have LoRA applied. For Qwen, `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]` is a good comprehensive set for attention and feed-forward networks.
    *   **Recommendation**: Keep this comprehensive set unless you have a specific reason to exclude layers.

#### 2. Training Parameters (`TrainingArguments`)

*   **`per_device_train_batch_size`**: The batch size processed on each GPU. Larger batch sizes can lead to more stable gradients and faster training, but require more GPU memory. If you encounter OOM (Out Of Memory) errors, reduce this value.
    *   **Recommendation**: Experiment with `4`, `8`, `16`. Current setting is `4`.

*   **`gradient_accumulation_steps`**: This allows you to simulate a larger batch size without increasing memory usage directly. Gradients are accumulated over this many steps before performing a single optimization step. The *effective batch size* is `per_device_train_batch_size * gradient_accumulation_steps`.
    *   **Recommendation**: If you reduce `per_device_train_batch_size` due to memory constraints, increase `gradient_accumulation_steps` to maintain a similar effective batch size. Current setting is `8`.

*   **`num_train_epochs`**: The total number of passes over the entire training dataset. More epochs can lead to better convergence, but also risk overfitting.
    *   **Recommendation**: Start with `3` (current setting) or `5`. Monitor validation loss if available, or evaluation metrics on a separate test set to prevent overfitting.

*   **`learning_rate`**: This is one of the most critical hyperparameters. It determines the step size at each iteration while moving towards a minimum of the loss function. A learning rate that is too high can cause the model to diverge, while one that is too low can lead to slow convergence.
    *   **Recommendation**: `2e-4` (current setting) is a good starting point for LoRA. You might try `1e-4` or `5e-5`.

*   **`warmup_steps`**: The number of steps during which the learning rate linearly increases from 0 to `learning_rate`. Warmup helps stabilize the training at the beginning.
    *   **Recommendation**: Current setting `100` is reasonable. You can adjust this based on your total training steps.

*   **`fp16` / `bf16`**: These enable mixed-precision training. `bf16` (bfloat16) is generally preferred if your GPU supports it, as it offers better numerical stability than `fp16` while still saving memory and speeding up computation. Unsloth automatically detects and uses the best available precision.
    *   **Recommendation**: Leave these as `fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported()` for optimal automatic handling.

*   **`save_steps`**: How frequently the model checkpoint is saved. Saving more often can be useful for recovery, but generates more files.
    *   **Recommendation**: Current setting is `50` (changed in previous turn from `100`). Adjust based on your needs.

*   **`save_total_limit`**: The maximum number of checkpoints to keep. Older checkpoints are removed when this limit is exceeded.
    *   **Recommendation**: Current setting is `5` (changed in previous turn from `3`). Keep this value to manage disk space.

By carefully tuning these parameters, especially `r`, `lora_alpha`, `per_device_train_batch_size`, `gradient_accumulation_steps`, `num_train_epochs`, and `learning_rate`, you can achieve better performance and more stable training for your specific task and dataset.

In [ ]:
# ==================== 4. RESUME TRAINING ====================
import torch
from datasets import load_from_disk
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Load the pre-processed datasets directly from Drive
data_save_path = "/content/drive/MyDrive/CSE4078_Project/dataset_splits_hf"
train_dataset = load_from_disk(f"{data_save_path}/train_dataset")
test_dataset = load_from_disk(f"{data_save_path}/test_dataset")
print("✅ Datasets loaded from disk!")

# 2. Re-initialize the Model and Tokenizer (Base Setup)
model_name = "Qwen/Qwen3.5-2B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)

# 3. Re-apply the exact same LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 4. Rebuild the Trainer exactly as before
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=4,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=3,
        output_dir="/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v1",
        push_to_hub=False,
        report_to="none",
        save_strategy="steps",
    ),
)

# 5. Resume training by pointing directly to the checkpoint
checkpoint_path = "/content/drive/MyDrive/CSE4078_Project/qwen35-2b-medical-v1/checkpoint-700"

print(f"🚀 Resuming fine-tuning from {checkpoint_path}...")
trainer.train(resume_from_checkpoint=checkpoint_path)